# 实验APL2-5数据分析

```
limit = 200000000
found = False

for n in range(2360, limit + 1):
    # Compute (4^n + 2^n + 1) mod n^2
    val = (pow(4, n, n*n) + pow(2, n, n*n) + 1) % (n*n)
    if val == 0:
        print("Next blue number found:", n)
        found = True
        break

if not found:
    print(f"No blue number found up to {limit}. Try increasing the limit.")

```
Next blue number found: 100170217

In [1]:
import numpy as np
from qutip import Qobj, fidelity
import matplotlib.pyplot as plt

import ysy_plot_utils as ypu
plt.rcParams.update(ypu.ysy_settings())

# 保真度测量

对于两比特纠缠态，通常使用 16 组测量基对纠缠态进行测量，可以重构出纠缠态的密
度矩阵。由密度矩阵可以计算其保真度。量子态保真度 F 的定义为：
$$ F(\rho, \sigma) = |\braket{\rho|\sigma}|^2 $$
即实验制备的量子态 ρ 和目标量子态 σ 投影的模平方。保真度约接近于 1 则说明纠缠源制
备的量子态越接近于最大纠缠态。其测试方法同 Bell 不等式相似，需要将测试端测量基旋
转至 16 组设定的位置，记录符合计数，并根据符合计数经公式获得其密度矩阵和保真度值。

In [2]:
# 1.1 定义单比特测量基、构造两比特投影测量算符

# 单比特计算基：|0> = [1, 0], |1> = [0, 1]
zero = np.array([1, 0], dtype=complex)
one  = np.array([0, 1], dtype=complex)

# 在计算基下定义实验中所用的测量基
# 这里我们假设: H = |0>, V = |1>
# D = (|0> + |1>)/√2, L = (|0> + i|1>)/√2
H = zero
V = one
D = (zero + one) / np.sqrt(2)
L = (zero + 1j*one) / np.sqrt(2)
single_qubit_basis = {
    'H': H,
    'V': V,
    'D': D,
    'L': L
}

# 构造两比特测量投影算符 P_{AB} = |A><A| \otimes |B><B|
# 其中 A, B ∈ {H, V, D, L}
def projector_2qubit(label1, label2):
    """
    返回 (|label1><label1|) \otimes (|label2><label2|)
    """
    ket1 = single_qubit_basis[label1]
    ket2 = single_qubit_basis[label2]
    # |A><A| for qubit1
    proj1 = np.outer(ket1, ket1.conj())
    # |B><B| for qubit2
    proj2 = np.outer(ket2, ket2.conj())
    return np.kron(proj1, proj2)  # 4x4 矩阵

In [3]:
# 1.2 读取实验测量数据

# 实验测到的 16 个基对的计数
counts = {
    'HH': 1800, 'HV': 10,   'HL': 800,  'HD': 1050, 
    'VD': 475,  'VL': 725,  'VH': 15,   'VV': 1150,
    'LV': 600,  'LH': 875,  'LL': 100,  'LD': 600,
    'DD': 1300, 'DL': 500,  'DH': 750,  'DV': 750
}

# 合计计数（总拍数）
total_counts = sum(counts.values())

# 先把所有的测量算符和对应的频率（概率）整理成列表，以便之后统一求解
measurement_ops = []
measured_probs  = []
# 我们固定一个遍历顺序(与 counts 的 key 吻合)，确保构造 A rho_vec = p 的矩阵时一一对齐
labels_order = list(counts.keys())
for key in labels_order:
    # 计算投影算符
    M = projector_2qubit(key[0], key[1])  # key[0]是第1比特，key[1]是第2比特
    measurement_ops.append(M)
    # 实验测量得到的频率
    freq = counts[key] / total_counts
    measured_probs.append(freq)
measured_probs = np.array(measured_probs, dtype=float)  # shape (16,)

## 量子断层扫描-密度矩阵重构

In [4]:
# 2. 构造线性方程用于 “线性反演” 重构密度矩阵

# 现在要建立 p = Tr(M_i * rho) = vec(M_i^T)^T * vec(rho)
# 若我们将 rho(4x4) 按行主序 flatten 成一个 16x1 向量 rho_vec，
# 则 p_i = (flatten(M_i^T))^T * rho_vec
# 故可令 A[i, :] = flatten(M_i^T)
A = np.zeros((16, 16), dtype=complex)
for i, M in enumerate(measurement_ops):
    M_flat = M.T.flatten()  # flatten(M^T)
    A[i, :] = M_flat

# 为了做伪逆，需要转为实部和虚部一致的 (16,16) 矩阵；但这里 M 是实对称投影，
# 整体应当都是实数(虚部=0)，可以直接用即可。
A = A.real  # 保证是实数 (投影矩阵没有虚数部分，此处简化处理)
p = measured_probs  # 16维实向量

# 使用 np.linalg.pinv 做伪逆
A_pinv = np.linalg.pinv(A)  # (16x16)
rho_vec_linear = A_pinv @ p  # 得到 16x1 的向量
# 重塑成 4x4 矩阵形式
rho_linear = rho_vec_linear.reshape((4,4))

# 线性反演结果一般不保证厄米、正定、以及 Tr(rho)=1，需要简单修正：
# 让其满足 Hermitian & Trace=1 作为最简单修正：
# 先 symmetrize: (rho + rho^\dagger)/2
# v3：修改为（加入对角化截断负特征值）：
rho_linear = 0.5 * (rho_linear + rho_linear.conj().T)   # 先厄米化
vals, vecs = np.linalg.eigh(rho_linear)
vals[vals < 0] = 0.0                                     # 截断负特征值
rho_linear = (vecs * vals) @ vecs.conj().T               # 恢复正定矩阵
rho_linear /= np.trace(rho_linear)    

In [5]:
# 3. 采用最大似然估计 (MLE) 重构密度矩阵

def mle_tomography(measurement_ops, counts_list, rho_init=None, max_iter=200, tol=1e-10):
    """
    基于常见迭代法的两比特 MLE。
    measurement_ops: list of 4x4 np.array (投影测量算符)
    counts_list: list or array, 与 measurement_ops 对应的测量结果计数
    rho_init: 初始猜测的密度矩阵 4x4
    max_iter: 最大迭代次数
    tol: 收敛阈值

    返回：rho_hat (满足正定、Tr(rho_hat)=1)
    """
    # 初始猜测：若不提供就用 I/4
    if rho_init is None:
        rho = np.eye(4, dtype=complex) / 4.0
    else:
        rho = 1.0 * rho_init

    # 总拍数
    N = sum(counts_list)

    for _ in range(max_iter):
        rho_old = rho.copy()
        
        # 计算所有投影结果在当前 rho 下的理论概率 p_i = Tr(M_i * rho)
        p_theory = np.array([np.real(np.trace(M_i @ rho)) for M_i in measurement_ops])
        
        # 若某个 p_i 太小，会导致分母接近 0，需要做个下限
        p_theory = np.maximum(p_theory, 1e-9)
        # v2：修改为（示例阈值从 1e-15 提高到 1e-9）：
        
        # 然后构造 Q = sum_i (n_i / p_i) * M_i
        # n_i 是该投影对应的实验计数
        Q = np.zeros((4,4), dtype=complex)
        for M_i, n_i, p_i in zip(measurement_ops, counts_list, p_theory):
            Q += (n_i / p_i) * M_i
        
        # 更新 rho：rho_{k+1} = (1 / Tr(Q rho Q)) Q rho Q
        # 保证了 rho_{k+1} 是正定且有归一化
        numerator = Q @ rho @ Q
        denom = np.trace(numerator)
        if abs(denom) < 1e-30:
            # 防止数值问题
            break
        # v2：修改为（加入厄米化、正定性修正与重新归一化）：
        rho = numerator / denom
        # 保证 Hermiticity
        rho = 0.5 * (rho + rho.conj().T)

        # 负特征值截断
        vals, vecs = np.linalg.eigh(rho)
        vals[vals < 0] = 0.0
        rho = (vecs * vals) @ vecs.conj().T

        # 归一化
        rho /= np.trace(rho)

        # 收敛判断
        if np.linalg.norm(rho - rho_old, ord='fro') < tol:
            break
    
    return rho

# 用上面的函数做 MLE 重构
counts_array = np.array(list(counts.values()), dtype=float)
rho_mle = mle_tomography(measurement_ops, counts_array, rho_init=rho_linear)

## 保真度计算

In [9]:
# 4. 计算保真度

# 目标 Bell 态：|\Phi^+> = (|00> + |11>)/√2
# 其密度矩阵 rho_ideal = |\Phi^+><\Phi^+|
bell_plus = np.zeros((4,1), dtype=complex)  # 4维向量表示 |00>,|01>,|10>,|11>
bell_plus[0,0] = 1.0 / np.sqrt(2)  # 对应 |00>
bell_plus[3,0] = 1.0 / np.sqrt(2)  # 对应 |11>
rho_ideal = bell_plus @ bell_plus.conj().T  # 4x4

# 转成 Qobj
rho_linear_qobj = Qobj(rho_linear)
rho_mle_qobj    = Qobj(rho_mle)
rho_ideal_qobj  = Qobj(rho_ideal)

f_linear = fidelity(rho_linear_qobj, rho_ideal_qobj)
f_mle    = fidelity(rho_mle_qobj,    rho_ideal_qobj)

In [10]:
# 5. 输出结果与简要评价
print("=============== 结果输出 ===============")
print("线性反演 (pseudo-inverse) 得到的密度矩阵：")
print(rho_linear)
print("线性反演与目标 Bell 态的保真度 =", f_linear)

print("\n最大似然估计 (MLE) 得到的密度矩阵：")
print(rho_mle)
print("MLE 与目标 Bell 态的保真度 =", f_mle)

=============== 结果输出 ===============
线性反演 (pseudo-inverse) 得到的密度矩阵：
[[ 0.57693056  0.04055578 -0.07629451  0.4337817 ]
 [ 0.04055578  0.00642378  0.00085137  0.02458588]
 [-0.07629451  0.00085137  0.02089899 -0.06776482]
 [ 0.4337817   0.02458588 -0.06776482  0.39574667]]
线性反演与目标 Bell 态的保真度 = 0.9592290242301028

最大似然估计 (MLE) 得到的密度矩阵：
[[0.43262947+3.00197001e-18j 0.20953842-1.38556679e-01j
  0.19244799-1.30793990e-01j 0.12972921-3.33742336e-01j]
 [0.20953842+1.38556679e-01j 0.14586223+1.01212256e-18j
  0.13509858-1.71373308e-03j 0.1697191 -1.20095821e-01j]
 [0.19244799+1.30793990e-01j 0.13509858+1.71373308e-03j
  0.12514935+7.80729194e-18j 0.15860597-1.09239528e-01j]
 [0.12972921+3.33742336e-01j 0.1697191 +1.20095821e-01j
  0.15860597+1.09239528e-01j 0.29635895-1.18213845e-17j]]
MLE 与目标 Bell 态的保真度 = 0.7030102658529623
